In [ ]:
# ===============================
# 1. Import Libraries
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

sns.set(style='whitegrid')
pd.set_option("display.max_columns", None)

# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("/mnt/data/train_and_test2 (2).csv")

print("First 5 rows:")
display(df.head())

# ===============================
# 3. Clean Dataset (IMPORTANT)
# ===============================

# Remove useless columns (zero columns)
zero_cols = [col for col in df.columns if "zero" in col]
df = df.drop(columns=zero_cols)

# Remove ID column
df = df.drop(columns=["Passengerid"])

print("\nColumns after cleaning:")
print(df.columns.tolist())

# ===============================
# 4. Define Target + Features
# ===============================
target_col = "2urvived"

# ===============================
# 5. Data Quality Check
# ===============================
print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

# ===============================
# 6. Handle Missing Values
# ===============================
df_imputed = df.copy()

# Age → mean
df_imputed['Age'] = df_imputed['Age'].fillna(df_imputed['Age'].mean())

# Embarked → mode
df_imputed['Embarked'] = df_imputed['Embarked'].fillna(df_imputed['Embarked'].mode()[0])

print("\nMissing values after fixing:")
print(df_imputed.isna().sum())

# ===============================
# 7. Outlier Detection (Fare)
# ===============================
print("\nOutlier detection on Fare")

plt.figure()
sns.boxplot(x=df_imputed['Fare'])
plt.show()

Q1 = df_imputed['Fare'].quantile(0.25)
Q3 = df_imputed['Fare'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df_no_outliers = df_imputed[
    (df_imputed['Fare'] >= lower) & (df_imputed['Fare'] <= upper)
]

print("Shape before:", df_imputed.shape)
print("Shape after:", df_no_outliers.shape)

# ===============================
# 8. Normalization
# ===============================
features = ['Age', 'Fare', 'sibsp', 'Parch', 'Pclass', 'Embarked']

# MinMax
minmax = MinMaxScaler()
df_minmax = df_no_outliers.copy()
df_minmax[features] = minmax.fit_transform(df_minmax[features])

print("\nMinMax scaled:")
display(df_minmax.head())

# Z-score
standard = StandardScaler()
df_zscore = df_no_outliers.copy()
df_zscore[features] = standard.fit_transform(df_zscore[features])

print("\nZ-score scaled:")
display(df_zscore.head())

# ===============================
# 9. Correlation + PCA
# ===============================
corr = df_zscore[features].corr()

plt.figure()
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Check correlation strength
corr_no_diag = corr.abs().copy()
np.fill_diagonal(corr_no_diag.values, 0)
max_corr = corr_no_diag.max().max()

print("Max correlation:", max_corr)

if max_corr >= 0.5:
    print("\nApplying PCA...")

    X = df_zscore[features]

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

    print("Explained variance ratio:")
    print(pca.explained_variance_ratio_)

    plt.figure()
    plt.scatter(pca_df['PC1'], pca_df['PC2'])
    plt.title("PCA Projection")
    plt.show()

else:
    print("\nPCA skipped.")

# ===============================
# 10. Final Summary
# ===============================
print("\nFINAL SUMMARY")
print("Original shape:", df.shape)
print("After cleaning:", df_imputed.shape)
print("After outlier removal:", df_no_outliers.shape)